# Three baseline classifiers: full-image + metadata

The goal is to train three separate full-image classifiers:

1. **Sagittal T2/STIR** → **Spinal Canal Stenosis**
2. **Sagittal T1** → **Neural Foraminal Narrowing**
3. **Axial T2** → **Subarticular Stenosis**

Each classifier receives:

```text
DICOM image slice + metadata(level, side, series_description) -> severity
```

The coordinate columns `x` and `y` are **not used here**. They will be used later for the ROI-guided experiment.


## Dataset preprocessing

In [1]:
# IMPORTS
import os
import copy
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
import pydicom

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [2]:
# SET SEED FOR REPRODUCIBILITY
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [3]:
# WORKING DIRECTORY
BASE_DIR = "/home/jupyter-lukj08@vse.cz/VSE_bachelor_thesis_lumbar_spine_degeneration_classification"

if os.path.exists(BASE_DIR):
    os.chdir(BASE_DIR)

print("Current directory:", os.getcwd())

Current directory: /home/jupyter-lukj08@vse.cz/VSE_bachelor_thesis_lumbar_spine_degeneration_classification


In [4]:
# LOAD MERGED RSNA DATASET
DATA_DIR = "./BC-data/data-rsna2024"
DATA_MERGED_PATH = os.path.join(DATA_DIR, "data_merged.csv")

data_merged = pd.read_csv(DATA_MERGED_PATH)

print(data_merged.shape)
display(data_merged.head())

(48692, 11)


,row_id,study_id,series_id,condition,level,series_description,instance_number,x,y,severity,img_path
0,4003253_spinal_canal_stenosis_l1_l2,4003253,702807833,Spinal Canal Stenosis,L1/L2,Sagittal T2/STIR,8,322.831858,227.964602,Normal/Mild,./BC-data/data-rsna2024/train_images/4003253/7...
1,4003253_spinal_canal_stenosis_l2_l3,4003253,702807833,Spinal Canal Stenosis,L2/L3,Sagittal T2/STIR,8,320.571429,295.714286,Normal/Mild,./BC-data/data-rsna2024/train_images/4003253/7...
2,4003253_spinal_canal_stenosis_l3_l4,4003253,702807833,Spinal Canal Stenosis,L3/L4,Sagittal T2/STIR,8,323.030303,371.818182,Normal/Mild,./BC-data/data-rsna2024/train_images/4003253/7...
3,4003253_spinal_canal_stenosis_l4_l5,4003253,702807833,Spinal Canal Stenosis,L4/L5,Sagittal T2/STIR,8,335.292035,427.327434,Normal/Mild,./BC-data/data-rsna2024/train_images/4003253/7...
4,4003253_spinal_canal_stenosis_l5_s1,4003253,702807833,Spinal Canal Stenosis,L5/S1,Sagittal T2/STIR,8,353.415929,483.964602,Normal/Mild,./BC-data/data-rsna2024/train_images/4003253/7...


In [5]:
# BASIC LABEL AND METADATA PREPARATION

SEVERITY_MAP = {
    "Normal/Mild": 0,
    "Moderate": 1,
    "Severe": 2,
}

SEVERITY_NAMES = ["Normal/Mild", "Moderate", "Severe"]

LEVELS = ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]

def determine_side(condition):
    if "Left" in condition:
        return "left"
    elif "Right" in condition:
        return "right"
    else:
        return "center"


def remove_side_from_condition(condition):
    return (
        condition
        .replace("Left ", "")
        .replace("Right ", "")
    )


data_merged["side"] = data_merged["condition"].apply(determine_side)
data_merged["base_condition"] = data_merged["condition"].apply(remove_side_from_condition)
data_merged["label_id"] = data_merged["severity"].map(SEVERITY_MAP)

# Keep only valid rows.
data_merged = data_merged.dropna(subset=[
    "study_id",
    "series_id",
    "instance_number",
    "img_path",
    "condition",
    "base_condition",
    "level",
    "side",
    "series_description",
    "severity",
    "label_id",
]).copy()

data_merged["label_id"] = data_merged["label_id"].astype(int)

display(data_merged[[
    "study_id",
    "series_id",
    "condition",
    "base_condition",
    "side",
    "level",
    "series_description",
    "severity",
    "label_id",
    "img_path",
]].head())

print(data_merged["severity"].value_counts())

,study_id,series_id,condition,base_condition,side,level,series_description,severity,label_id,img_path
0,4003253,702807833,Spinal Canal Stenosis,Spinal Canal Stenosis,center,L1/L2,Sagittal T2/STIR,Normal/Mild,0,./BC-data/data-rsna2024/train_images/4003253/7...
1,4003253,702807833,Spinal Canal Stenosis,Spinal Canal Stenosis,center,L2/L3,Sagittal T2/STIR,Normal/Mild,0,./BC-data/data-rsna2024/train_images/4003253/7...
2,4003253,702807833,Spinal Canal Stenosis,Spinal Canal Stenosis,center,L3/L4,Sagittal T2/STIR,Normal/Mild,0,./BC-data/data-rsna2024/train_images/4003253/7...
3,4003253,702807833,Spinal Canal Stenosis,Spinal Canal Stenosis,center,L4/L5,Sagittal T2/STIR,Normal/Mild,0,./BC-data/data-rsna2024/train_images/4003253/7...
4,4003253,702807833,Spinal Canal Stenosis,Spinal Canal Stenosis,center,L5/S1,Sagittal T2/STIR,Normal/Mild,0,./BC-data/data-rsna2024/train_images/4003253/7...


severity
Normal/Mild    37626
Moderate        7950
Severe          3081
Name: count, dtype: int64


In [6]:
def resolve_img_path(row):
    original = str(row["img_path"])

    candidates = [
        original,
        original.replace("./BC-data/", "./"),
        original.replace("BC-data/data-rsna2024", "data-rsna2024"),
        os.path.join(
            DATA_DIR,
            "train_images",
            str(row["study_id"]),
            str(row["series_id"]),
            f"{int(row['instance_number'])}.dcm"
        ),
    ]

    for path in candidates:
        if os.path.exists(path):
            return path

    # Return the most likely path even if it does not exist,
    # so that later error messages are informative.
    return candidates[-1]


data_merged["img_path"] = data_merged.apply(resolve_img_path, axis=1)

missing_paths = (~data_merged["img_path"].apply(os.path.exists)).sum()
print("Missing image paths:", missing_paths)

display(data_merged[["study_id", "series_id", "instance_number", "img_path"]].head())

Missing image paths: 0


,study_id,series_id,instance_number,img_path
0,4003253,702807833,8,./BC-data/data-rsna2024/train_images/4003253/7...
1,4003253,702807833,8,./BC-data/data-rsna2024/train_images/4003253/7...
2,4003253,702807833,8,./BC-data/data-rsna2024/train_images/4003253/7...
3,4003253,702807833,8,./BC-data/data-rsna2024/train_images/4003253/7...
4,4003253,702807833,8,./BC-data/data-rsna2024/train_images/4003253/7...


## Split to prevent data leakage

We split by `study_id`, not by image row. This avoids having observations from the same MRI study in both training and validation/test sets.


In [7]:
# SPLIT BY STUDY_ID TO PREVENT DATA LEAKAGE
study_ids = data_merged["study_id"].unique()

train_ids, temp_ids = train_test_split(
    study_ids,
    test_size=0.30,
    random_state=42,
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
)

train_df = data_merged[data_merged["study_id"].isin(train_ids)].copy()
val_df   = data_merged[data_merged["study_id"].isin(val_ids)].copy()
test_df  = data_merged[data_merged["study_id"].isin(test_ids)].copy()

print("Training Data Size:  ", train_df.shape[0])
print("Validation Data Size:", val_df.shape[0])
print("Test Data Size:      ", test_df.shape[0])

print("\nUnique studies:")
print("Train:", train_df["study_id"].nunique())
print("Val:  ", val_df["study_id"].nunique())
print("Test: ", test_df["study_id"].nunique())

Training Data Size:   34045
Validation Data Size: 7306
Test Data Size:       7306

Unique studies:
Train: 1381
Val:   296
Test:  297


In [8]:
# DEFINE THE THREE CLASSIFIERS
MODEL_CONFIGS = {
    "spinal_canal_stenosis": {
        "base_condition": "Spinal Canal Stenosis",
        "series_description": "Sagittal T2/STIR",
    },
    "neural_foraminal_narrowing": {
        "base_condition": "Neural Foraminal Narrowing",
        "series_description": "Sagittal T1",
    },
    "subarticular_stenosis": {
        "base_condition": "Subarticular Stenosis",
        "series_description": "Axial T2",
    },
}


def make_model_df(df, base_condition, series_description):
    out = df[
        (df["base_condition"] == base_condition) &
        (df["series_description"] == series_description)
    ].copy()

    out = out.dropna(subset=[
        "img_path",
        "label_id",
        "level",
        "side",
        "series_description",
    ])

    out = out[out["img_path"].apply(os.path.exists)].copy()

    return out


model_dfs = {}

for model_name, cfg in MODEL_CONFIGS.items():
    model_dfs[model_name] = {
        "train": make_model_df(
            train_df,
            cfg["base_condition"],
            cfg["series_description"],
        ),
        "val": make_model_df(
            val_df,
            cfg["base_condition"],
            cfg["series_description"],
        ),
        "test": make_model_df(
            test_df,
            cfg["base_condition"],
            cfg["series_description"],
        ),
    }

for model_name in model_dfs:
    print("\n", model_name)
    print("Train:", len(model_dfs[model_name]["train"]))
    print("Val:  ", len(model_dfs[model_name]["val"]))
    print("Test: ", len(model_dfs[model_name]["test"]))
    print("Train severity counts:")
    print(model_dfs[model_name]["train"]["severity"].value_counts())


 spinal_canal_stenosis
Train: 6820
Val:   1464
Test:  1464
Train severity counts:
severity
Normal/Mild    5964
Moderate        513
Severe          343
Name: count, dtype: int64

 neural_foraminal_narrowing
Train: 13779
Val:   2950
Test:  2960
Train severity counts:
severity
Normal/Mild    10800
Moderate        2462
Severe           517
Name: count, dtype: int64

 subarticular_stenosis
Train: 13441
Val:   2892
Test:  2882
Train severity counts:
severity
Normal/Mild    9601
Moderate       2549
Severe         1291
Name: count, dtype: int64


In [9]:
# METADATA ENCODINGS

level_to_id = {level: i for i, level in enumerate(LEVELS)}

side_to_id = {
    "center": 0,
    "left": 1,
    "right": 2,
}

series_to_id = {
    "Sagittal T2/STIR": 0,
    "Sagittal T1": 1,
    "Axial T2": 2,
}

print("level_to_id:", level_to_id)
print("side_to_id:", side_to_id)
print("series_to_id:", series_to_id)

level_to_id: {'L1/L2': 0, 'L2/L3': 1, 'L3/L4': 2, 'L4/L5': 3, 'L5/S1': 4}
side_to_id: {'center': 0, 'left': 1, 'right': 2}
series_to_id: {'Sagittal T2/STIR': 0, 'Sagittal T1': 1, 'Axial T2': 2}


In [ ]:
# TRANSFORMS
# MRI images are grayscale, converted to 3 channels for ImageNet-pretrained models.


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(5),
    transforms.RandomAffine(degrees=0, translate=(0.02, 0.02)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [ ]:
# DICOM READING

def read_dicom_as_pil(path):
    ds = pydicom.dcmread(path)
    img = ds.pixel_array.astype(np.float32)

    # Invert MONOCHROME1 if needed.
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = img.max() - img

    # Robust intensity clipping.
    low, high = np.percentile(img, [1, 99])
    img = np.clip(img, low, high)

    # Normalize to 0-255 uint8.
    img = img - img.min()
    img = img / (img.max() + 1e-6)
    img = (img * 255).astype(np.uint8)

    # Convert grayscale to RGB-like 3-channel PIL image.
    image = Image.fromarray(img).convert("RGB")
    return image

In [ ]:
# CUSTOM DATASET DEFINITION

class LumbarMetadataClassificationDataset(Dataset):
    def __init__(
        self,
        df,
        level_to_id,
        side_to_id,
        series_to_id,
        transform=None,
    ):
        self.df = df.reset_index(drop=True)
        self.level_to_id = level_to_id
        self.side_to_id = side_to_id
        self.series_to_id = series_to_id
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = read_dicom_as_pil(row["img_path"])

        if self.transform is not None:
            image = self.transform(image)

        metadata = {
            "level": torch.tensor(self.level_to_id[row["level"]], dtype=torch.long),
            "side": torch.tensor(self.side_to_id[row["side"]], dtype=torch.long),
            "series": torch.tensor(self.series_to_id[row["series_description"]], dtype=torch.long),
        }

        label = torch.tensor(int(row["label_id"]), dtype=torch.long)

        return image, metadata, label

## Training setup

In [ ]:
# DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# BUILD MODEL
# ResNet34 backbone + metadata embeddings.
# This keeps the spirit of your original baseline model, but adds metadata.

class ResNet34MetadataClassifier(nn.Module):
    def __init__(
        self,
        n_classes=3,
        n_levels=5,
        n_sides=3,
        n_series=3,
        level_emb_dim=8,
        side_emb_dim=4,
        series_emb_dim=4,
        hidden_dim=256,
        dropout=0.3,
    ):
        super().__init__()

        self.backbone = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        image_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.level_emb = nn.Embedding(n_levels, level_emb_dim)
        self.side_emb = nn.Embedding(n_sides, side_emb_dim)
        self.series_emb = nn.Embedding(n_series, series_emb_dim)

        metadata_features = level_emb_dim + side_emb_dim + series_emb_dim

        self.classifier = nn.Sequential(
            nn.Linear(image_features + metadata_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, images, metadata):
        image_features = self.backbone(images)

        level_features = self.level_emb(metadata["level"])
        side_features = self.side_emb(metadata["side"])
        series_features = self.series_emb(metadata["series"])

        metadata_features = torch.cat(
            [level_features, side_features, series_features],
            dim=1,
        )

        features = torch.cat(
            [image_features, metadata_features],
            dim=1,
        )

        outputs = self.classifier(features)
        return outputs


def build_model():
    model = ResNet34MetadataClassifier(
        n_classes=3,
        n_levels=len(level_to_id),
        n_sides=len(side_to_id),
        n_series=len(series_to_id),
    )
    return model.to(device)

In [ ]:
# CLASS-WEIGHTED LOSS
# Handles imbalance between Normal/Mild, Moderate and Severe.

def make_weighted_criterion(df):
    y = df["label_id"].values
    counts = np.bincount(y, minlength=3)

    # Avoid division by zero.
    counts = np.maximum(counts, 1)

    weights = len(y) / (3 * counts)
    weights = torch.tensor(weights, dtype=torch.float32).to(device)

    print("Class counts:", counts)
    print("Class weights:", weights.detach().cpu().numpy())

    return nn.CrossEntropyLoss(weight=weights)


def move_metadata_to_device(metadata, device):
    return {
        key: value.to(device)
        for key, value in metadata.items()
    }

In [ ]:
# TRAINING FUNCTION

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, metadata, labels in loader:
        images = images.to(device)
        metadata = move_metadata_to_device(metadata, device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images, metadata)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_bal_acc = balanced_accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_bal_acc, epoch_f1

In [ ]:
# EVALUATION FUNCTION

@torch.no_grad()
def evaluate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, metadata, labels in loader:
        images = images.to(device)
        metadata = move_metadata_to_device(metadata, device)
        labels = labels.to(device)

        outputs = model(images, metadata)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_bal_acc = balanced_accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return (
        epoch_loss,
        epoch_acc,
        epoch_bal_acc,
        epoch_f1,
        np.array(all_labels),
        np.array(all_preds),
    )

In [ ]:
# FULL TRAINING LOOP

def fit_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    model_name,
    epochs=10,
):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_f1 = -np.inf

    history = {
        "train_loss": [],
        "train_acc": [],
        "train_bal_acc": [],
        "train_f1": [],
        "val_loss": [],
        "val_acc": [],
        "val_bal_acc": [],
        "val_f1": [],
    }

    start_time = time.time()
    best_path = f"best_{model_name}_full_image_metadata_resnet34.pth"

    for epoch in range(epochs):
        train_loss, train_acc, train_bal_acc, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        val_loss, val_acc, val_bal_acc, val_f1, _, _ = evaluate_one_epoch(
            model, val_loader, criterion, device
        )

        scheduler.step(val_f1)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["train_bal_acc"].append(train_bal_acc)
        history["train_f1"].append(train_f1)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_bal_acc"].append(val_bal_acc)
        history["val_f1"].append(val_f1)

        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
            f"Train Bal Acc: {train_bal_acc:.4f}, Train F1: {train_f1:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, "
            f"Val Bal Acc: {val_bal_acc:.4f}, Val F1: {val_f1:.4f}"
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, best_path)
            print(f"  -> Best model saved with Val F1: {best_val_f1:.4f}")

    elapsed = time.time() - start_time
    print(f"\nTraining complete in {elapsed/60:.2f} minutes")
    print(f"Best Val F1: {best_val_f1:.4f}")
    print(f"Best model path: {best_path}")

    model.load_state_dict(best_model_wts)
    return model, history, best_path

In [ ]:
# PLOTTING FUNCTION

def plot_history(history, title="Training history"):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(18, 4))

    plt.subplot(1, 4, 1)
    plt.plot(epochs, history["train_loss"], label="Train")
    plt.plot(epochs, history["val_loss"], label="Val")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.legend()

    plt.subplot(1, 4, 2)
    plt.plot(epochs, history["train_acc"], label="Train")
    plt.plot(epochs, history["val_acc"], label="Val")
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.legend()

    plt.subplot(1, 4, 3)
    plt.plot(epochs, history["train_bal_acc"], label="Train")
    plt.plot(epochs, history["val_bal_acc"], label="Val")
    plt.title("Balanced Accuracy")
    plt.xlabel("Epoch")
    plt.legend()

    plt.subplot(1, 4, 4)
    plt.plot(epochs, history["train_f1"], label="Train")
    plt.plot(epochs, history["val_f1"], label="Val")
    plt.title("Macro F1")
    plt.xlabel("Epoch")
    plt.legend()

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

## Train the three baseline classifiers

In [ ]:
# TRAIN A SINGLE CLASSIFIER

def train_single_classifier(
    model_name,
    train_part,
    val_part,
    batch_size=32,
    epochs=10,
    lr=1e-4,
):
    print("=" * 80)
    print("Training classifier:", model_name)
    print("Train samples:", len(train_part))
    print("Val samples:  ", len(val_part))
    print("=" * 80)

    train_dataset = LumbarMetadataClassificationDataset(
        train_part,
        level_to_id=level_to_id,
        side_to_id=side_to_id,
        series_to_id=series_to_id,
        transform=train_transform,
    )

    val_dataset = LumbarMetadataClassificationDataset(
        val_part,
        level_to_id=level_to_id,
        side_to_id=side_to_id,
        series_to_id=series_to_id,
        transform=val_transform,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    model = build_model()

    criterion = make_weighted_criterion(train_part)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
    )

    model, history, best_path = fit_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        model_name=model_name,
        epochs=epochs,
    )

    return model, history, best_path

In [ ]:
# RUN TRAINING FOR ALL THREE CLASSIFIERS

trained_models = {}
histories = {}
best_paths = {}

for model_name in MODEL_CONFIGS.keys():
    train_part = model_dfs[model_name]["train"]
    val_part = model_dfs[model_name]["val"]

    model, history, best_path = train_single_classifier(
        model_name=model_name,
        train_part=train_part,
        val_part=val_part,
        batch_size=32,
        epochs=10,
        lr=1e-4,
    )

    trained_models[model_name] = model
    histories[model_name] = history
    best_paths[model_name] = best_path

    plot_history(history, title=model_name)

## Evaluate models

In [ ]:
# CONFUSION MATRIX AND REPORT

def show_confusion_and_report(y_true, y_pred, class_names, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.tight_layout()
    plt.show()

    print(classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        labels=list(range(len(class_names))),
        zero_division=0,
    ))

In [ ]:
# TEST EVALUATION FOR A SINGLE CLASSIFIER

def evaluate_single_classifier(
    model_name,
    test_part,
    best_path,
    batch_size=32,
):
    test_dataset = LumbarMetadataClassificationDataset(
        test_part,
        level_to_id=level_to_id,
        side_to_id=side_to_id,
        series_to_id=series_to_id,
        transform=val_transform,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    model = build_model()
    model.load_state_dict(torch.load(best_path, map_location=device))

    criterion = make_weighted_criterion(test_part)

    test_loss, test_acc, test_bal_acc, test_f1, y_true, y_pred = evaluate_one_epoch(
        model,
        test_loader,
        criterion,
        device,
    )

    print("=" * 80)
    print("Test results:", model_name)
    print(f"Loss:              {test_loss:.4f}")
    print(f"Accuracy:          {test_acc:.4f}")
    print(f"Balanced Accuracy: {test_bal_acc:.4f}")
    print(f"Macro F1:          {test_f1:.4f}")
    print("=" * 80)

    show_confusion_and_report(
        y_true,
        y_pred,
        SEVERITY_NAMES,
        title=f"{model_name} test confusion matrix",
    )

    return {
        "model": model_name,
        "input": "full_image + metadata",
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "test_balanced_accuracy": test_bal_acc,
        "test_macro_f1": test_f1,
    }

In [ ]:
# RUN TEST EVALUATION FOR ALL THREE CLASSIFIERS

test_results = []

for model_name in MODEL_CONFIGS.keys():
    result = evaluate_single_classifier(
        model_name=model_name,
        test_part=model_dfs[model_name]["test"],
        best_path=best_paths[model_name],
        batch_size=32,
    )
    test_results.append(result)

results_table = pd.DataFrame(test_results)
display(results_table)

results_table.to_csv("full_image_metadata_three_classifiers_results.csv", index=False)

## Notes for the next experiment

This notebook trains the **full-image + metadata** version.

For the next ROI-guided version, keep the same:

- split,
- metadata,
- model,
- loss,
- training loop,
- evaluation metrics.

Only replace the dataset image-loading step with:

```text
DICOM image -> crop around (x, y) -> resize -> model
```

That gives a clean comparison:

```text
full image + metadata
vs.
ground-truth ROI crop + metadata
```
